Read video

In [ ]:
import cv2

cap = cv2.VideoCapture("dynamic_raw/good_morning.mov")
frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)

cap.release()

print(f"Total frames in video: {len(frames)}")


Total frames in video: 244


Extract Landmarks
- Each norm_keypoints = flattened vector of 63 floats (21 points × 3).
- landmark_sequences = sequence of vectors for the video.

In [ ]:
import mediapipe as mp
from fsl_preprocessing import extract_keypoints_from_hand_landmarks, normalize_landmarks

mp_hands = mp.solutions.hands.Hands(static_image_mode=False,
                                    max_num_hands=1,
                                    min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

landmark_sequences = []

for frame in frames:
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = mp_hands.process(rgb_frame)
    
    if result.multi_hand_landmarks:
        lm = result.multi_hand_landmarks[0]  # only first hand
        keypoints = extract_keypoints_from_hand_landmarks(lm)
        norm_keypoints = normalize_landmarks(keypoints, scale_mode='bbox')
        landmark_sequences.append(norm_keypoints)


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [ ]:
print("Sequence length:", len(landmark_sequences))
print("Landmark vector shape:", np.array(landmark_sequences).shape)

Sequence length: 119
Landmark vector shape: (119, 63)


Fix Sequence Length
- Now every video has shape (30, 63).
- Can be fed into LSTM.

In [ ]:
import numpy as np
import importlib
import fsl_dynamic_utils
importlib.reload(fsl_dynamic_utils)
from fsl_dynamic_utils import pad_or_truncate_sequence


SEQ_LENGTH = 30  # all sequences will be 30 frames

# Check that there are actually frames
if len(landmark_sequences) == 0:
    raise ValueError("No landmarks detected in this video!")

# Apply padding/truncation
X_video = pad_or_truncate_sequence(landmark_sequences, length=SEQ_LENGTH)

# Convert to numpy array
X_video = np.array(X_video)  # shape: (30, 63)

# Optional debug
print("Final sequence shape:", X_video.shape)
print("Min / Max values:", X_video.min(), X_video.max())


Final sequence shape: (30, 63)
Min / Max values: -1.0 0.37636905718994595


In [ ]:
mp_draw.draw_landmarks(frame, lm, mp.solutions.hands.HAND_CONNECTIONS)
cv2.imshow("Landmarks", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()


### Train LSTM (Subset of signs only)


In [ ]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [ ]:
# -------------------------------
# 1️⃣ Configuration
# -------------------------------
VIDEO_DIR = "dynamic_raw"        # your dataset folder
SEQ_LENGTH = 30
SCALE_MODE = "bbox"

# Load CSV mapping: id -> label
csv_file = "csv/labels.csv"  # your CSV file
df = pd.read_csv(csv_file)
id_to_label = dict(zip(df["id"].astype(str), df["label"]))




In [ ]:
# -------------------------------
# 2️⃣ Loop through selected folders (DEBUG MODE)
# -------------------------------

X_data = []
y_data = []

mp_hands = mp.solutions.hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5
)

subset_ids = ["0", "3", "7"]  # Only process these folders
MAX_VIDEOS_PER_CLASS = 5     # Limit to first 5 videos per sign

for folder_id in subset_ids:

    folder_path = os.path.join(VIDEO_DIR, folder_id)

    if not os.path.isdir(folder_path):
        print(f"Skipping non-directory: {folder_path}")
        continue

    # Get label from CSV
    if folder_id not in id_to_label:
        print(f"Warning: folder ID {folder_id} not in CSV")
        continue

    label = id_to_label[folder_id]

    video_files = sorted(os.listdir(folder_path))[:MAX_VIDEOS_PER_CLASS]

    print(f"\nProcessing class {label} (ID {folder_id})")

    for video_file in video_files:
        video_path = os.path.join(folder_path, video_file)

        cap = cv2.VideoCapture(video_path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            print(f"Warning: no frames in {video_path}")
            continue

        # Extract landmarks
        landmark_sequences = []

        for frame in frames:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = mp_hands.process(rgb_frame)

            if result.multi_hand_landmarks:
                lm = result.multi_hand_landmarks[0]
                keypoints = extract_keypoints_from_hand_landmarks(lm)
                norm_keypoints = normalize_landmarks(
                    keypoints,
                    scale_mode=SCALE_MODE
                )
                landmark_sequences.append(norm_keypoints)
            else:
                landmark_sequences.append(np.zeros(63).tolist())

        # Pad / truncate
        seq = pad_or_truncate_sequence(
            landmark_sequences,
            length=SEQ_LENGTH
        )

        X_data.append(seq)
        y_data.append(label)

        print(f"✔ Processed {video_file} → final shape: {len(seq)} frames")



Processing class GOOD MORNING (ID 0)


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✔ Processed 0.MOV → final shape: 30 frames
✔ Processed 1.MOV → final shape: 30 frames
✔ Processed 10.MOV → final shape: 30 frames
✔ Processed 11.MOV → final shape: 30 frames
✔ Processed 12.MOV → final shape: 30 frames

Processing class HELLO (ID 3)
✔ Processed 0.MOV → final shape: 30 frames
✔ Processed 1.MOV → final shape: 30 frames
✔ Processed 10.MOV → final shape: 30 frames
✔ Processed 11.MOV → final shape: 30 frames
✔ Processed 12.MOV → final shape: 30 frames

Processing class THANK YOU (ID 7)
✔ Processed 0.MOV → final shape: 30 frames
✔ Processed 1.MOV → final shape: 30 frames
✔ Processed 10.MOV → final shape: 30 frames
✔ Processed 11.MOV → final shape: 30 frames
✔ Processed 12.MOV → final shape: 30 frames


In [ ]:
# -------------------------------
# 3️⃣ Convert to arrays
# -------------------------------
X_data = np.array(X_data)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)


In [ ]:
# -------------------------------
# 4️⃣ Train / Val / Test Split
# -------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)


print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)


Train shape: (10, 30, 63) (10,)
Val shape: (2, 30, 63) (2,)
Test shape: (3, 30, 63) (3,)


In [ ]:

# -------------------------------
# 5️⃣ Save artifacts
# -------------------------------
os.makedirs("models/dynamic", exist_ok=True)

np.save("models/dynamic/X_train.npy", X_train)
np.save("models/dynamic/X_val.npy", X_val)
np.save("models/dynamic/X_test.npy", X_test)
np.save("models/dynamic/y_train.npy", y_train)
np.save("models/dynamic/y_val.npy", y_val)
np.save("models/dynamic/y_test.npy", y_test)

with open("models/dynamic/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

preprocess_config = {
    "scale_mode": SCALE_MODE,
    "seq_length": SEQ_LENGTH
}

with open("models/dynamic/preprocess_config.pkl", "wb") as f:
    pickle.dump(preprocess_config, f)

print("Dynamic dataset processed and saved successfully!")


Dynamic dataset processed and saved successfully!


In [ ]:
import numpy as np
import pickle

# Load dynamic dataset
X_train = np.load("models/dynamic/X_train.npy")
y_train = np.load("models/dynamic/y_train.npy")

with open("models/dynamic/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Number of classes:", len(le.classes_))
print("Classes:", le.classes_)


print("One sample shape:", X_train[0].shape)
print("Min value:", X_train.min())
print("Max value:", X_train.max())



X_train shape: (10, 30, 63)
y_train shape: (10,)
Number of classes: 3
Classes: ['GOOD MORNING' 'HELLO' 'THANK YOU']
One sample shape: (30, 63)
Min value: 0.0
Max value: 0.0


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

num_classes = len(le.classes_)

model = Sequential([
    LSTM(64, input_shape=(30, 63)),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=50, batch_size=4)


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.4500 - loss: 1.0991
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4187 - loss: 1.0970
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4187 - loss: 1.0973
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5125 - loss: 1.0925
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3875 - loss: 1.0958
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4187 - loss: 1.0944
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4812 - loss: 1.0919
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3562 - loss: 1.0979
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3562 - loss: 1.0977
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4500 - loss: 1.0910
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2937 - loss: 1.1031    
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4500 - loss: 1.08

In [ ]:
from collections import Counter
print("Class distribution:", Counter(y_train))


Class distribution: Counter({np.int64(0): 4, np.int64(1): 3, np.int64(2): 3})


In [ ]:
model.save("models/dynamic/subset_lstm_model.h5")

In [ ]:
print("Sequence shape:", np.array(sequence).shape)


Sequence shape: (0,)


### Train ALL DATASET

In [ ]:
# -------------------------------
# 1️⃣ Configuration
# -------------------------------
VIDEO_DIR = "dynamic_raw"        # your dataset folder
SEQ_LENGTH = 30
SCALE_MODE = "bbox"

# Load CSV mapping: id -> label
csv_file = "csv/labels"  # your CSV file
df = pd.read_csv(csv_file)
id_to_label = dict(zip(df["id"].astype(str), df["label"]))




In [ ]:
# -------------------------------
# 2️⃣ Loop through all folders
# -------------------------------

X_data = []
y_data = []

mp_hands = mp.solutions.hands.Hands(static_image_mode=False,
                                    max_num_hands=1,
                                    min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

for folder_id in sorted(os.listdir(VIDEO_DIR)):
    folder_path = os.path.join(VIDEO_DIR, folder_id)
    if not os.path.isdir(folder_path):
        continue

    # Get label from CSV
    if folder_id not in id_to_label:
        print(f"Warning: folder ID {folder_id} not in CSV")
        continue
    label = id_to_label[folder_id]

    video_files = sorted(os.listdir(folder_path))
    for video_file in video_files:
        video_path = os.path.join(folder_path, video_file)
        cap = cv2.VideoCapture(video_path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
        cap.release()

        if len(frames) == 0:
            print(f"Warning: no frames in {video_path}")
            continue

        # Extract landmarks
        landmark_sequences = []
        for frame in frames:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = mp_hands.process(rgb_frame)
            if result.multi_hand_landmarks:
                lm = result.multi_hand_landmarks[0]
                keypoints = extract_keypoints_from_hand_landmarks(lm)
                norm_keypoints = normalize_landmarks(keypoints, scale_mode=SCALE_MODE)
                landmark_sequences.append(norm_keypoints)
            else:
                landmark_sequences.append(np.zeros(63).tolist())

        # Pad / truncate
        seq = pad_or_truncate_sequence(landmark_sequences, length=SEQ_LENGTH)
        X_data.append(seq)
        y_data.append(label)

        print(f"Processed video {video_path} → sequence length {len(seq)}")


In [2]:
import numpy as np

X_train = np.load("models/dynamic/X_train.npy")
y_train = np.load("models/dynamic/y_train.npy")

print("Shape:", X_train.shape)
print("Min:", X_train.min())
print("Max:", X_train.max())
print("Unique values count:", len(np.unique(X_train)))

Shape: (10, 30, 63)
Min: 0.0
Max: 0.0
Unique values count: 1


In [3]:
print(X_train[0][0][:10])   # first 10 values of first frame

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [5]:
import cv2
import mediapipe as mp

video_path = "C:\Projects\signia-fsl-recognition\dynamic_raw\3\0.MOV"

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False)

cap = cv2.VideoCapture(video_path)

frame_count = 0
detect_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    if results.multi_hand_landmarks:
        detect_count += 1

cap.release()

print("Total frames:", frame_count)
print("Frames with detection:", detect_count)

Total frames: 0
Frames with detection: 0
